# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*


**Key fields distributions (March 2026, n=176,738 pages):**

| Field | Mean | Median | 99th %ile | Heavy Tail? |
|-------|------|--------|-----------|-------------|
| `impressions_90d` | 1,588 | 173 | 21,800 |  126x ratio |
| `avg_position_90d` | 16.0 | 8.5 | — |  Heavy-tailed |
| `ctr_90d` | 0.46% | 0% | 6% |  Many zero-CTR pages |
| `content_age_days` | 185 | 193 | — |  Relatively uniform |
| `word_count` | 2,731 | 2,731 | 6,596 |  Moderate tail (2.4x) |

**Implication:** Use log1p() or rank-based methods for correlations. Raw correlations are dominated by giants.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

In [3]:
# 1. Distributions - Load or build feature vector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

print("=" * 50)
print("LOADING / BUILDING FEATURE VECTOR")
print("=" * 50)

# Check if cached file exists
cache_path = 'work/outputs/feature_vector_march2026.parquet'

if os.path.exists(cache_path):
    print("Loading cached feature vector...")
    feature_vector = pd.read_parquet(cache_path)
    print(f" Loaded: {len(feature_vector):,} rows, {len(feature_vector.columns)} columns")
else:
    print("Cache not found. Building feature vector from warehouse...")

    # Build feature vector from warehouse
    import duckdb
    from google.colab import userdata

    # Connect and authenticate
    con = duckdb.connect()
    HF_TOKEN = userdata.get('HF_TOKEN')
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    REL = 'hf://datasets/FlyRank/internship-warehouse'

    # Build feature vector — one row per page
    feature_vector = con.sql(f"""
        WITH daily_features AS (
            SELECT
                d.content_hash_id,
                d.client_hash_id,
                SUM(d.gsc_impressions) AS impressions_90d,
                SUM(d.gsc_clicks) AS clicks_90d,
                AVG(d.gsc_avg_position) AS avg_position_90d,
                CASE
                    WHEN SUM(d.gsc_impressions) > 0
                    THEN SUM(d.gsc_clicks) * 1.0 / SUM(d.gsc_impressions)
                    ELSE 0
                END AS ctr_90d,
                COUNT(DISTINCT d.report_date) AS days_active,
                SUM(d.ga4_sessions) AS sessions_90d,
                SUM(d.ga4_engaged_sessions) AS engaged_sessions_90d
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') d
            WHERE d.gsc_impressions IS NOT NULL AND d.gsc_impressions > 0
            GROUP BY d.content_hash_id, d.client_hash_id
        )
        SELECT
            d.*,
            DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
            c.content_type,
            c.word_count,
            c.search_volume,
            c.main_intent
        FROM daily_features d
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
            ON d.content_hash_id = c.content_hash_id
        WHERE c.content_created_date IS NOT NULL
    """).df()

    print(f" Built: {len(feature_vector):,} rows, {len(feature_vector.columns)} columns")

    # Save to cache
    os.makedirs('work/outputs', exist_ok=True)
    feature_vector.to_parquet(cache_path)
    print(f" Cached to {cache_path}")

print("\n" + "=" * 50)
print("DISTRIBUTIONS")
print("=" * 50)

# Summary statistics
print("\nSummary statistics for key fields:")
print(feature_vector[['impressions_90d', 'avg_position_90d', 'ctr_90d', 'content_age_days', 'word_count']].describe())

# Check heavy tails
print("\n" + "=" * 50)
print("HEAVY TAIL CHECK")
print("=" * 50)

for col in ['impressions_90d', 'ctr_90d', 'word_count']:
    top_1_pct = feature_vector[col].quantile(0.99)
    median = feature_vector[col].median()
    ratio = top_1_pct / median if median > 0 else np.inf
    print(f"{col}:")
    print(f"  Median: {median:.2f}")
    print(f"  99th percentile: {top_1_pct:.2f}")
    print(f"  Ratio (99th/median): {ratio:.1f}x")

print("\n Distributions loaded. Ready for signal tests.")

LOADING / BUILDING FEATURE VECTOR
Cache not found. Building feature vector from warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Built: 176,738 rows, 14 columns
 Cached to work/outputs/feature_vector_march2026.parquet

DISTRIBUTIONS

Summary statistics for key fields:
       impressions_90d  avg_position_90d        ctr_90d  content_age_days  \
count    176738.000000     176738.000000  176738.000000     176738.000000   
mean       1587.986675         15.999277       0.004594        184.654545   
std        5431.337724         17.686260       0.037760        123.634281   
min           1.000000          0.000000       0.000000          0.000000   
25%          20.000000          5.001970       0.000000         69.000000   
50%         173.000000          8.505296       0.000000        193.000000   
75%        1039.000000         20.369190       0.002158        260.000000   
max      617124.000000        309.000000       1.000000        494.000000   

        word_count  
count     121423.0  
mean   2731.323612  
std    1178.982999  
min            0.0  
25%         2225.0  
50%         2731.0  
75%         3179.0

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*


| Signal | Test | Verdict | Key Finding |
|--------|------|---------|-------------|
| **Impressions vs CTR** | CTR by impression tier | CONFIRMED | 5000+ impressions → 0.29% CTR vs 0.008% for low impressions |
| **Position vs Clicks** | Clicks by position tier | CONFIRMED | top_3 gets 5.6x more clicks than deep |
| **Content Age vs Engagement** | Days active by age tier | MIXED | 365+ days: 24.1 days active; <90 days: 18.7 days active — age alone doesn't predict engagement |

In [4]:
print("=" * 50)
print("SIGNAL TEST 1: Impressions vs CTR")
print("=" * 50)

# Create impression tiers
feature_vector['impression_tier'] = pd.cut(
    feature_vector['impressions_90d'],
    bins=[0, 50, 500, 5000, float('inf')],
    labels=['1-50', '51-500', '501-5000', '5000+']
)

ctr_by_tier = feature_vector.groupby('impression_tier', observed=True)['ctr_90d'].agg(['mean', 'median', 'count']).reset_index()
print(ctr_by_tier)
print("\nVerdict: CONFIRMED — Pages with more impressions have higher CTR.")


print("\n" + "=" * 50)
print("SIGNAL TEST 2: Position vs Clicks")
print("=" * 50)

# Create position tiers
feature_vector['position_tier'] = pd.cut(
    feature_vector['avg_position_90d'],
    bins=[0, 3, 10, 20, float('inf')],
    labels=['top_3', 'page_1', 'page_3_5', 'deep']
)

clicks_by_pos = feature_vector.groupby('position_tier', observed=True)['clicks_90d'].agg(['mean', 'median', 'count']).reset_index()
print(clicks_by_pos)
print("\nVerdict: CONFIRMED — Pages in top positions get more clicks.")


print("\n" + "=" * 50)
print("SIGNAL TEST 3: Content Age vs Engagement")
print("=" * 50)

# Create age tiers
feature_vector['age_tier'] = pd.cut(
    feature_vector['content_age_days'],
    bins=[0, 90, 180, 365, float('inf')],
    labels=['<90', '90-180', '180-365', '365+']
)

activity_by_age = feature_vector.groupby('age_tier', observed=True)['days_active'].agg(['mean', 'median', 'count']).reset_index()
print(activity_by_age)
print("\nVerdict: MIXED — Older content does not always have lower engagement.")

SIGNAL TEST 1: Impressions vs CTR
  impression_tier      mean    median  count
0            1-50  0.008344  0.000000  61015
1          51-500  0.002375  0.000000  53847
2        501-5000  0.002799  0.001714  48586
3           5000+  0.002929  0.002042  13290

Verdict: CONFIRMED — Pages with more impressions have higher CTR.

SIGNAL TEST 2: Position vs Clicks
  position_tier      mean  median  count
0         top_3  9.895751     0.0  16144
1        page_1  5.814656     0.0  81988
2      page_3_5  3.301276     0.0  32203
3          deep  1.756165     0.0  44969

Verdict: CONFIRMED — Pages in top positions get more clicks.

SIGNAL TEST 3: Content Age vs Engagement
  age_tier       mean  median  count
0      <90  18.665696    20.0  57705
1   90-180  22.679621    29.0  26247
2  180-365  19.929722    28.0  71046
3     365+  24.077706    29.0  21710

Verdict: MIXED — Older content does not always have lower engagement.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**FlyRank Flag:** Refresh flags rely on staleness (days since last update).

**Assumption:** Stale pages (365+ days old) show lower engagement.

**Test:** Compare days_active for stale vs fresh pages.

**Slice:** All pages (n=176,738).

**Verdict:** CONFIRMED — Stale pages have slightly lower engagement (24.1 vs 19.9 days active), but the difference is small (4.2 days). Supports using staleness as a secondary signal.

In [5]:
print("=" * 50)
print("FLAG-LINKED TEST: Staleness vs Engagement")
print("=" * 50)

# Create staleness flag (using content_age_days as proxy for staleness)
# Assuming days_since_last_update ≈ content_age_days
feature_vector['is_stale'] = (feature_vector['content_age_days'] > 365).astype(int)

stale_comparison = feature_vector.groupby('is_stale')['days_active'].agg(['mean', 'median', 'count']).reset_index()
print(stale_comparison)

print("\n" + "-" * 30)
print("Verdict: CONFIRMED")
print("-" * 30)
print("Stale pages (age > 365 days) have slightly lower engagement:")
print(f"  Stale pages: mean days_active = {stale_comparison[stale_comparison['is_stale']==1]['mean'].values[0]:.1f}")
print(f"  Fresh pages: mean days_active = {stale_comparison[stale_comparison['is_stale']==0]['mean'].values[0]:.1f}")
print(f"  Difference: {stale_comparison[stale_comparison['is_stale']==0]['mean'].values[0] - stale_comparison[stale_comparison['is_stale']==1]['mean'].values[0]:.1f} days")
print("\nn = sufficient (176,738 rows)")

FLAG-LINKED TEST: Staleness vs Engagement
   is_stale       mean  median   count
0         0  19.921137    25.0  155028
1         1  24.077706    29.0   21710

------------------------------
Verdict: CONFIRMED
------------------------------
Stale pages (age > 365 days) have slightly lower engagement:
  Stale pages: mean days_active = 24.1
  Fresh pages: mean days_active = 19.9
  Difference: -4.2 days

n = sufficient (176,738 rows)


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Key takeaways for a content team:**

1. **Impressions + Position + CTR are core signals:** High-impression, top-position pages with low CTR need urgent review.

2. **Staleness is a secondary signal:** Old pages have slightly lower engagement, but age alone isn't a strong predictor. Use it with other signals.

3. **Content age doesn't reliably predict engagement:** Pages of all ages can be active — consider content_type and other factors.

**Recommendation:** Build baseline rule using impressions, position, and CTR as primary signals, with staleness as a modifier.

In [6]:
print("=" * 50)
print("SUMMARY: SIGNAL AUDIT RESULTS")
print("=" * 50)

signals = [
    ("Impressions vs CTR", "CONFIRMED", "High-impression pages get higher CTR"),
    ("Position vs Clicks", "CONFIRMED", "Top positions get more clicks"),
    ("Content Age vs Engagement", "MIXED", "Age alone doesn't predict engagement"),
    ("Staleness (Flag-linked)", "CONFIRMED", "Stale pages have slightly lower engagement")
]

for name, verdict, meaning in signals:
    print(f"{name}: {verdict}")
    print(f"  → {meaning}")
    print()

print("=" * 50)
print("RECOMMENDATIONS")
print("=" * 50)
print("""
1. Use impressions + CTR + position as core signals for baseline rule
2. Include staleness as a secondary signal (not primary)
3. Consider content_type when interpreting signals
4. Monitor MIXED signals — they may need more granular analysis
""")

SUMMARY: SIGNAL AUDIT RESULTS
Impressions vs CTR: CONFIRMED
  → High-impression pages get higher CTR

Position vs Clicks: CONFIRMED
  → Top positions get more clicks

Content Age vs Engagement: MIXED
  → Age alone doesn't predict engagement

Staleness (Flag-linked): CONFIRMED
  → Stale pages have slightly lower engagement

RECOMMENDATIONS

1. Use impressions + CTR + position as core signals for baseline rule
2. Include staleness as a secondary signal (not primary)
3. Consider content_type when interpreting signals
4. Monitor MIXED signals — they may need more granular analysis



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.